# 4. RQ5: Security-sensitive capabilities and their drift across copies

**What we measure.** Not vulnerabilities and not attacks, but *security-sensitive capabilities*: instructions that give the agent the power to cause harm if the skill were malicious, tampered with in a copy, or misused. Each capability maps to the closest CWE weakness class and a severity tier.

**Methodology step 3: quantitative + statistical modelling.**

* RQ5.1 Prevalence of each capability, also weighted by copies (exposure).
* RQ5.2 Drift: within each variant family, which capabilities an edited copy **adds** or **removes** relative to the most-copied member. Additions vs removals are tested with an exact McNemar test.
* RQ5.3 Logistic regression: which variants add capabilities?

**Methodology step 4 (validation)** happens in notebook 05: manual labelling to measure detector precision.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
print("database:", os.environ["GITSKILLS_DB"])
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint
import statsmodels.formula.api as smf

In [ ]:
skills = clean.strict(clean.load_skills())
risk_cols = [c for c in skills.columns if c.startswith("risk_")]
pd.DataFrame({"capability": [c[5:] for c in risk_cols],
              "severity": [patterns.RISK_SEVERITY[c[5:]] for c in risk_cols],
              "closest CWE": [patterns.RISK_CWE[c[5:]] for c in risk_cols]})

## 4.1 Prevalence and exposure

In [ ]:
prev = pd.DataFrame({
    "severity": [patterns.RISK_SEVERITY[c[5:]] for c in risk_cols],
    "share_of_skills": skills[risk_cols].mean().values,
    "share_of_occurrences": (skills[risk_cols].mul(skills["copies"], axis=0).sum() / skills["copies"].sum()).values,
}, index=[c[5:] for c in risk_cols]).sort_values(["severity", "share_of_skills"], ascending=[True, False])
save_table(prev.reset_index(names="capability"), "rq5_risk_prevalence")
high = [c for c in risk_cols if patterns.RISK_SEVERITY[c[5:]] == "high"]
print(f"skills with >=1 high-severity capability: {skills[high].any(axis=1).mean():.1%}")
prev

## 4.2 Drift across edited copies

In [ ]:
fam = reuse.name_families(skills)
drift = reuse.variant_drift(fam, risk_cols)
k, n = int(drift["any_added"].sum()), len(drift)
lo, hi = proportion_confint(k, n, method="wilson")
print(f"variants adding >=1 capability: {k:,}/{n:,} = {k/n:.1%} (95% CI {lo:.1%}-{hi:.1%})")
rows = []
for c in risk_cols:
    a, r = int(drift[f"added_{c}"].sum()), int(drift[f"removed_{c}"].sum())
    rows.append({"capability": c[5:], "severity": patterns.RISK_SEVERITY[c[5:]], "added": a, "removed": r,
                 "mcnemar_p": binomtest(a, a + r, 0.5).pvalue if a + r else np.nan})
mc = pd.DataFrame(rows)
save_table(mc, "rq5_mcnemar_added_vs_removed")
mc

## 4.3 Which variants add capabilities? (logistic regression)
Outcome: the variant adds at least one capability absent from its family reference. Predictors: reference popularity (log copies), family size, cross-owner copy, bundled scripts, assistant authorship.

In [ ]:
d = drift.join(skills.set_index("file_sha")[["n_owners", "has_scripts", "first_commit_ai"]], on="file_sha")
d["family_size"] = d.groupby("family")["file_sha"].transform("size") + 1
d = d.assign(added=d["any_added"].astype(int), log_ref_copies=np.log1p(d["ref_copies"]),
             log_family_size=np.log(d["family_size"]), cross_owner=(d["n_owners"] > 1).astype(int),
             has_scripts=d["has_scripts"].fillna(0).astype(int), ai_authored=d["first_commit_ai"].notna().astype(int))
model = smf.logit("added ~ log_ref_copies + log_family_size + cross_owner + has_scripts + ai_authored", data=d).fit(disp=False)
odds = pd.DataFrame({"odds_ratio": np.exp(model.params), "ci_low": np.exp(model.conf_int()[0]),
                     "ci_high": np.exp(model.conf_int()[1]), "p": model.pvalues})
save_table(odds.reset_index(names="term"), "rq5_logit_odds_ratios")
odds.round(3)